# Project 2: Model Analysis Project

## Table of contents <a id='toc0_'></a>
- [Question 1: A Consumer with Two Nests](#toc1_)
    - [Question 1.1: The problem in nested budget shares](#toc1_1_)
    - [Question 1.2: Calibration](#toc1_2_)
    - [Question 1.3: How do you know your answer is right?](#toc1_3_)
- [Question 2: Solving the Model Numerically](#toc2_)
    - [Question 2.1: A Two-dimensional Grid Search](#toc2_1_)
    - [Question 2.2: L-BFGS-B](#toc2_2_)
- [Question 3: Relative prices and demand](#toc3_)

We import the nessesary packages and the two python files given in the project description:

In [ ]:
%reload_ext autoreload

# Import consumer.py and government.py - vil i dog hellere sætte ind i vores egen?
from Consumer import ConsumerClass
from Government import GovernmentClass

# Import all necessary packages
import numpy as np
import pandas as pd
from scipy import optimize
from types import SimpleNamespace

# plotting
import matplotlib.pyplot as plt
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.rcParams.update({'axes.grid':True,'grid.color':'black','grid.alpha':'0.25','grid.linestyle':'--'})
plt.rcParams.update({'font.size': 14})

# autoreload modules when code is run
%load_ext autoreload
%autoreload 2

# The functions
import modelproject

## Question 1: A Consumer with Two Nests <a id='toc1_'></a>

### Question 1.1: The problem in nested budget shares <a id='toc1_1_'></a>

To be able to start the project, we fill out the pass arguments in consumer.py, based on the model description. We have clearly marked in consumer.py where we change the code.

In [ ]:
# we check whether the code runs smoothly after we have filled it in
#Print the parameters
model=ConsumerClass()
print(model)

#We know that with a CES function, inserting x1=x2=x3=1 means that utility should also be 1, as the function collapses:
assert np.isclose(model.utility(1.0,1.0,1.0), 1.0), 'utility(1,1,1) should be 1'
print(' \n utility sanity check passed')

#Could also add a check on whether corner solutions are stable??

So we find that the code we have inserted in consumption.py runs smoothly and holds the basic values that we expected

### Question 1.2: Calibration <a id='toc1_2_'></a>

We now implement the two calibrations that we will use in the following questions, where only the substitution between the modes of transport differ (complements or substitutes respectively). To do this, we let model_comp be where $sigma_B=0.4$, so complements, and let model_sub have the value of $sigma_B=3.0$ so bus and train are substitutes.

In [ ]:
model_comp = ConsumerClass()
model_sub  = ConsumerClass(par={'sigma_B':3.0})

print(model_comp)
print(model_sub)

### Question 1.3: How do you know your answer is right? <a id='toc1_3_'></a>

#### Question 1.3.1

As we cannot check the solution for the model, we check whether the solutions are possible, meaning that the budget shares sum to 1 and are all positive.

In [ ]:
#First, check if the solution is possible
for model in (model_comp, model_sub):
    sol = model.solve(do_print=False)
    assert 0 < sol.s1 < 1 and 0 < sol.s2 and 0 < sol.s3
    assert np.isclose(sol.s1+sol.s2+sol.s3, 1.0), 'the budget shares do not sum to one'

print('the answer is possible')

#### Question 1.3.2

To check the answer, we compare the results we get from the model when solved using optimize.minimize and when we solve the model numerically in section 2. Here, we have inserted the grid-solution from section 2.

In [ ]:
# Write code here...


We find that...

## Question 2: Solving the Model Numerically <a id='toc2_'></a>

### Question 2.1: A Two-dimensional Grid Search <a id='toc2_1_'></a>

#### Question 2.1.1

We make the grid of the N values in the consumer.py file in solve_grid - we now report the values the weigths and the three implied budget shares for each calibration.

In [ ]:
print('when transport are complements:')
sol_grid_comp = model_comp.solve_grid()
print('when transport are substitutes:')
sol_grid_sub  = model_sub.solve_grid()

So we see that when the budget shares for train and bus are more equal when the modes of transport are complements, whereas when they are substitutes, more are spent on bus trips, and results in a higher utility for the consumer.

#### Question 2.1.2

We now plot the utility over the nested shares in a 3D plot and a contour plot with the solution shown for both calibrations:

In [ ]:
from matplotlib import cm # import the colormap
#Define the figure, so only have to write it out once:
#Because we need the contour plot for 2.2.2, we write it as a seperate function, then call it:

def plot_contour(sol, title, ax=None, path=None): #last two calls needed for 2.2.2
    if ax is None: #so the plot works for 2.1.2, without the ax call
        fig, ax = plt.subplots(figsize=(6.5,5.5))
    else:
        fig = ax.figure

    ax.contourf(sol.s1_grid, sol.w_grid, sol.u_grid, levels=30)
    ax.plot(sol.s1, sol.w, 'o', color='red', ms=10, label='solution')

    if path is not None:
        ax.plot(path[:,0], path[:,1], '-o', color='white', ms=4, lw=1.5, label='convergence path')

    ax.set_title(f'Contour plot, with the solution marked')
    ax.set_xlabel('$s_1$')
    ax.set_ylabel('$w$')
    ax.legend()

    return fig, ax

def plot_utility_surf(sol, title):
    fig = plt.figure(figsize=(13,5.5))

    #The 3d plot:
    ax1 = fig.add_subplot(1,2,1,projection='3d')
    surf = ax1.plot_surface(sol.s1_grid, sol.w_grid, sol.u_grid, cmap='viridis') #make sure the colourmap matches the utility values
    fig.colorbar(surf, ax=ax1, shrink=0.6)
    # labels and titles:
    fig.suptitle(title, fontsize=16, fontweight='bold')
    ax1.set_title('Utility over the nested shares')
    ax1.set_xlabel('$s_1$')
    ax1.set_ylabel('$w$')
    ax1.set_zlabel('$u$')
    #change the direction of the y-axis and where the z-axis is placed
    ax1.view_init(azim=-135,elev=30) 
    ax1.set_box_aspect([4,4,3],zoom=0.8)

    #and now the contour plot:
    ax2=fig.add_subplot(1,2,2)
    plot_contour(sol, title, ax=ax2)


    fig.tight_layout(pad=0.1)


plot_utility_surf(sol_grid_comp, 'Complements')

plot_utility_surf(sol_grid_sub, 'Substitutes')

So we see that when means of transportation are substitutes, the weight on the bus, w, is higher compared to when it is complements, as here the consumer wants to spread the weight on both trains and busses. We also see that the utility surface seem to be flatter for subsitutes, meaning that a wider array of w-values can lead to almost the same utility-value compared to when it is complements, where the utility surface is more pointed.

#### Question 2.1.3

We now test how sensitive the grid solving method is to the value of N, meaning how fine the grid is. The finer the grid, the more evaluations of u.

In [ ]:
#for the 4 N-values and the two model calibrations:
N_values=[50, 100, 500, 1000]
models={'Complements': model_comp, 'Substitutes': model_sub}

results={}
for name, model in models.items():
    for N in N_values:
        opt=model.solve_grid(N=N, do_print=False)
        results[(name, N)]= opt
        print(f'{name}, N={N:4d}: s1={opt.s1:.4f}, w={opt.w:.4f}, u={opt.u:.6f}, evaluations={N*N}')
    print() 

#To see how much the answer moves, we look at the differences in s1, w and u, every time N becomes larger:
print('Changes in the nested budget share and the utility with a finer grid search')
for name in models:
    print(name)
    for N_prev, N in zip(N_values, N_values[1:]):
        opt_prev = results[(name,N_prev)]
        opt = results[(name,N)]
        print(f'  N={N:4d}: Δs1={abs(opt.s1-opt_prev.s1):.5f}, Δw={abs(opt.w-opt_prev.w):.5f}, Δu={abs(opt.u-opt_prev.u):.6f}')


We see that the number of evaluations of u grows quadratically with N, since the grid search evaluates every point on an N×N grid. With a finer grid, the answer does move, and the answer improves smoothly: it becomes more accurate as the grid gets finer, but only slightly once N is already large. This higher accuracy comes at a high computational cost, since the number of evaluations grows rapidly with N. Comparing the two calibrations, the improvement in u with a finer grid is small for both, but the improvement in w converges more slowly for substitutes than for complements. As we saw in 2.1.2, many w-values give almost the same u-value under substitutes (flatter utility surface), so the grid search converges more slowly onto the exact best w — even though the resulting utility is still accurate.

#### Question 2.1.4

All in all, we find that for both calibrations the surface is steep along the s1-direction (food), so there is a high cost to utility when moving food away from optimum. However, in the w-direction, the two different calibrations are quite different where the surface is steep when means of transportation are complements, indicating a peak around the optimum. For substitutes, the surface is more flat in the w-direction, indication a more flat plateau around optimum. Therefore we expect that it is harder for an optimizer to find the answer when busses and trains are substitutes, as the utility only changes slightly for different w-values when close to optimum, making it harder to find out what direction improves utility. This was shown very clearly in 2.1.3, where the improvement in w converged more slowly for subsitutes than for complements as the grid was refined.

### Question 2.2: L-BFGS-B <a id='toc2_2_'></a>

#### Question 2.2.1

We implement the L-BFGS-B solution in the consumer.py file, calling it solve. We compare the solution, the number of function evaluations and the running time with the grid search, that we made in 2.1    

In [ ]:
# We import time, and run the solutions for both model calibration.
import time
sol_min_comp = model_comp.solve(do_print=False)
sol_min_sub  = model_sub.solve(do_print=False)
models = {'Complements': model_comp, 'Substitutes': model_sub}

for name, model in models.items():
    print(f'--{name}--')

    # grid search
    t0 = time.perf_counter()
    sol_grid = model.solve_grid(do_print=False) 
    t_grid = time.perf_counter() - t0

    # L-BFGS-B
    t0 = time.perf_counter()
    sol_min = model.solve(do_print=False)
    t_min = time.perf_counter() - t0


    print(f'  grid search: s1={sol_grid.s1:.4f}, w={sol_grid.w:.4f}, u={sol_grid.u:.6f}, '
          f'evaluations=40000, time={t_grid:.4f}s') # evaluations are 40.000 since N=200 if we do not change it
    print(f'  L-BFGS-B:    s1={sol_min.s1:.4f}, w={sol_min.w:.4f}, u={sol_min.u:.6f}, '
          f'evaluations={sol_min.res.nfev}, time={t_min:.4f}s')
    print()



So we find that the methods find slightly different solutions (but agree at least on the first two decimals), and that the grid search takes slightly longer, with a much higher count of evaluations. So the two methods do converge towards the same answer, but they do not agree. This is because the grid is 200 x 200, and therefore perhaps not fine enough. We also see that L-BFGS-B needs only few evaluations, because it uses a quasi-Newton method with bounds - this is a lot more targeted than the grid search.

#### Question 2.2.2

We now record the convergence path, seeing how the quasi-Newton method with bounds find the answer. We plot this in the contour plot we made in 2.1.2 but also plot how far each of the iterations are from the solving-point on a log-scaled iteration:

In [ ]:
#Define a figure with both plots:
def plot_convergence_full(sol_grid, sol_min, title):

    fig = plt.figure(figsize=(13,5.5))
    fig.suptitle(title, fontsize=16, fontweight='bold')

    # contour plot with the solve path overlaid
    ax1 = fig.add_subplot(1,2,1)
    plot_contour(sol_grid, title, ax=ax1, path=sol_min.path)

    # distance from the final point, using a log scale
    ax2 = fig.add_subplot(1,2,2)
    path = sol_min.path
    final = path[-1]
    distances = np.linalg.norm(path - final, axis=1)

    ax2.plot(distances, '-o', ms=4)
    ax2.set_yscale('log')
    ax2.set_title(f'Distance from final point')
    ax2.set_xlabel('iteration, $k$')
    ax2.set_ylabel('distance (log scale)')

    fig.tight_layout(pad=0.1)

plot_convergence_full(sol_grid_comp, sol_min_comp, 'Complements')

plot_convergence_full(sol_grid_sub, sol_min_sub, 'Substitutes')

We find that...

#### Question 2.2.3

We look at...

In [ ]:
%who

We find that...

#### Question 2.2.4

We look at...

In [ ]:
# Write code here...

We find that...

#### Question 2.2.5

We look at...

In [ ]:
# Write code here...

We find that...

## Question 3: Relative prices and demand <a id='toc3_'></a>

#### Question 3.3.1

We look at...

In [ ]:
# Write code here...

We find that...

#### Question 3.3.2

We look at...

In [ ]:
# Write code here...

We find that...

#### Question 3.3.3

We look at...

In [ ]:
# Write code here...

We find that...

#### Question 3.3.4

We look at...

In [ ]:
# Write code here...

We find that...

## Question 4: Lump-sum taxes and product taxes <a id='toc3_'></a>

#### Question 4.3.1

We start by 